## Streaming

By default, `model.invoke()` waits until the **whole** answer is ready and then returns it in one piece. With **streaming**, the model sends the answer back **in small chunks as it is generated**. Your app can show text immediately instead of waiting.

### Why use streaming?

- **Faster feel:** the first words appear in about a second, even if the full answer takes 20 seconds.
- **Better chat UX:** the "typing" effect users expect from ChatGPT-style apps.
- **Long outputs:** you can show, log or stop a response before it finishes.

### `invoke()` vs `stream()`

| | `invoke()` | `stream()` |
|---|---|---|
| Returns | One `AIMessage` | An iterator of `AIMessageChunk` |
| When you get output | After the full response is done | While it is being generated |
| Use for | Scripts, batch jobs, tool pipelines | Chat UIs, live output |

### Basic example

```python
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:llama-3.3-70b-versatile")

for chunk in model.stream("Explain what an AI guardrail is in 3 sentences"):
    print(chunk.text, end="", flush=True)
```

- `model.stream(...)` returns a generator. Each `chunk` is a small piece of the reply.
- `chunk.text` gives just the text. Use it instead of `chunk.content`, which can be a list of blocks for models like Gemini.
- `end=""` stops `print` from adding a newline after every chunk.
- `flush=True` makes the text appear right away.

### Combining chunks into the full message

Chunks can be added together with `+`. This gives you the full response after streaming, which is useful for saving it or reading token usage:

```python
full = None
for chunk in model.stream("Tell me a joke"):
    full = chunk if full is None else full + chunk
    print(chunk.text, end="", flush=True)

print()
print(full.text)             # complete text
print(full.usage_metadata)   # token counts (if the provider sends them)
```

### Async streaming

In async code such as FastAPI or async notebooks, use `astream()`:

```python
async for chunk in model.astream("Tell me a joke"):
    print(chunk.text, end="", flush=True)
```

### Streaming from an agent

Agents can stream too. `stream_mode="messages"` gives you the model's tokens as they are generated:

```python
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in Paris?"}]},
    stream_mode="messages",
):
    print(token.content_blocks)
```

Other modes include `"updates"` (one update per step) and `"values"` (the full state after each step).

### Key takeaways

1. Use `.stream()` when you want output to appear as it is generated.
2. Each chunk is small. Read its text with `chunk.text`.
3. Add chunks together (`full = full + chunk`) to rebuild the complete message.
4. Use `.astream()` in async code.
5. Streaming changes how you receive the answer, not the answer itself.


In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [4]:
## Without streaming we need to wait until the resposne gets generted 

## Chat google generative ai 

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
response=model.invoke("Write me a 200 words paragrah on Artificial Intelligence")
print(response.text)

Artificial Intelligence (AI) has rapidly transitioned from the realm of science fiction into the cornerstone of modern technological innovation, fundamentally reshaping how we live, work, and interact with the world. At its core, AI refers to the simulation of human intelligence in machines programmed to think, learn, adapt, and solve complex problems. By processing vast amounts of data at unprecedented speeds through advanced algorithms and machine learning, AI systems can recognize patterns, make predictions, and continuously improve without explicit human intervention. Today, its applications are ubiquitous, powering everything from personalized recommendations on streaming platforms and voice-activated virtual assistants to autonomous vehicles, medical diagnostics, and sophisticated financial trading systems. While AI holds immense potential to revolutionize industries, optimize global supply chains, and tackle grand challenges like climate change and disease eradication, it also i

In [6]:
## Using streaming

for chunk in model.stream("Write me a 200 words paragrah on Artificial Intelligence"):
    print(chunk.text, end="|", flush=True)


Artificial Intelligence| (AI) has rapidly transformed from a staple of science fiction into a foundational pillar of modern technology|, fundamentally reshaping how we live, work, and interact with the world. At its core, AI refers to the simulation of human intelligence| in machines programmed to think, learn, adapt, and solve complex problems. Through advancements in machine learning and deep learning, algorithms can now| analyze vast oceans of data at unprecedented speeds, recognizing patterns and making autonomous decisions that once required exclusively human cognition. Today, AI’|s impact is ubiquitous. In healthcare, it accelerates drug discovery and detects diseases like cancer with staggering precision. In finance, it| detects fraudulent transactions in milliseconds, while in daily life, it powers personalized recommendations, natural language assistants, and autonomous navigation systems|. However, this technological leap is not without its challenges. The rise of AI brings f

In [7]:
## Using streaming

for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)


Parrots| have colorful feathers primarily for **survival, communication, and evolution**. While to human| eyes they look like they’re wearing bright, tropical costumes, in the wild, those colors serve very practical purposes. 

Here| are the main reasons why parrots are so colorful:

### 1. Camouflage in the Rainforest
This sounds counter|intuitive—how does bright red or green hide a bird? It all depends on the environment:
* **Green Par|rots:** Most parrots (like many Amazon parrots and parakeets) are green. In the lush, leafy canopy of a| tropical rainforest, green feathers act as **cryptic camouflage**, making it very difficult for predators like hawks and eagles to spot them against| the foliage.
* **Bright Colors (Reds, Blues, Yellows):** For birds that live in mixed| light or open canopy areas (like Macaws), bright patches of color can actually break up their silhouette, making it harder| for a predator to recognize them as a distinct bird shape among the shifting shadows and sun

In [9]:
model.invoke("Write me a 200 words paragrah on Artificial Intelligence")

AIMessage(content=[{'type': 'text', 'text': 'Artificial Intelligence (AI) has rapidly transformed from a niche concept of science fiction into the defining technological frontier of the twenty-first century. At its core, AI refers to the simulation of human intelligence in machines programmed to think, learn, reason, and solve problems autonomously. By processing colossal amounts of data at unprecedented speeds, modern machine learning algorithms can recognize patterns, make predictions, and adapt to new information without explicit human intervention. This capability has revolutionized nearly every facet of modern life. In healthcare, AI accelerates drug discovery and detects diseases like cancer with astonishing precision. In finance, it monitors fraudulent transactions in milliseconds, while in transportation, it paves the way for autonomous vehicles. Moreover, generative AI models now create art, compose music, and draft complex code, blurring the line between human and machine cre

## Batch

By default, calling `model.invoke()` in a loop sends prompts **one after another**, so the total time is the sum of every call. With **batching**, you send a **list of independent prompts** and the model runs them **in parallel**. Your app gets all the answers much faster.

### Why use batch?

- **Faster for many prompts:** total time is roughly the time of the slowest call, not the sum of all calls.
- **Less code:** one call replaces a `for` loop.
- **Bulk jobs:** classify, summarize or evaluate many items at once.

### `invoke()` vs `batch()`

| | `invoke()` | `batch()` |
|---|---|---|
| Input | One prompt | A list of prompts |
| Returns | One `AIMessage` | A list of `AIMessage`, in the same order as the inputs |
| Execution | One call at a time | Calls run in parallel |
| Use for | A single question, chat turns | Bulk jobs, many independent prompts |

### Basic example

```python
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:llama-3.3-70b-versatile")

questions = [
    "What is an AI guardrail?",
    "What is prompt injection?",
    "What is a jailbreak?",
]

responses = model.batch(questions)

for q, r in zip(questions, responses):
    print(q, "->", r.text)
```

- `model.batch(...)` takes a **list** of prompts and returns a **list** of responses.
- Results come back in the **same order** as the inputs, even though the calls ran in parallel.
- `r.text` gives just the text of each response.

### Limiting parallel calls

Providers enforce rate limits. If you send too many prompts at once, you may get `429` errors. Cap how many calls run at the same time with `max_concurrency`:

```python
responses = model.batch(questions, config={"max_concurrency": 2})  # 2 calls at a time
```

### Handling failures

By default, if one call fails, the whole `batch()` raises an error. Use `return_exceptions=True` to get the errors back as results and keep the successful ones:

```python
responses = model.batch(questions, return_exceptions=True)

for q, r in zip(questions, responses):
    if isinstance(r, Exception):
        print(q, "-> FAILED:", r)
    else:
        print(q, "->", r.text)
```

### Async batching

In async code such as FastAPI or async notebooks, use `abatch()`:

```python
responses = await model.abatch(questions)
```

### Batching with an agent

Agents can be batched too. Pass a list of inputs, one per conversation:

```python
responses = agent.batch([
    {"messages": [{"role": "user", "content": "What is the weather in Paris?"}]},
    {"messages": [{"role": "user", "content": "What is the weather in Tokyo?"}]},
])

for r in responses:
    print(r["messages"][-1].text)
```

### Key takeaways

1. Use `.batch()` when you have many **independent** prompts.
2. Input is a list. Output is a list in the **same order**.
3. Set `max_concurrency` to stay under provider rate limits.
4. Use `return_exceptions=True` so one failure doesn't lose the rest.
5. Use `.abatch()` in async code.
6. Batching runs prompts faster. It doesn't change the answers.


In [12]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])

for response in responses:
    print(response.text)

Parrots have colorful feathers primarily for **survival, communication, and evolution**. While humans see their bright reds, greens, blues, and yellows as dazzling decoration, in the wild, these colors serve very practical purposes. 

Here are the main reasons why parrots are so colorful:

### 1. Camouflage in the Rainforest
It might seem strange that bright colors help parrots hide, but in their natural habitat—the dense, sun-dappled rainforest—it works. 
* **Green** is the most common color among parrots because it provides perfect camouflage against the green leaves of the canopy. Predators like hawks have a hard time spotting a green parrot sitting still in a tree.
* Other bright colors (like reds, yellows, and blues) often mimic tropical flowers, fruits, or the way sunlight filters through the leaves, breaking up the parrot’s silhouette and hiding it from predators.

### 2. Finding a Mate
For parrots, color is a sign of health and genetic quality. During courtship, vibrant feather

In [ ]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
],
config={'max_concurrency':5}) ## sends 5 request at a time -> n concurrency at a time 

for response in responses:
    print(response.text)

Parrots have colorful feathers primarily for **survival, communication, and evolution**. While to human eyes their bright greens, reds, blues, and yellows might seem like they would easily spot predators, in their natural habitats, these colors actually serve several vital purposes:

### 1. Camouflage in the Rainforest
It sounds contradictory, but bright colors can be good camouflage. Most parrots live in tropical rainforests, which are filled with bright sunlight, deep shadows, and colorful fruits, flowers, and leaves. 
* A bright green parrot sitting in the canopy is almost invisible against the green leaves.
* Splashes of red, blue, or yellow often break up the bird’s silhouette, mimicking patches of sunlight or colorful tropical flowers and fruit. 

### 2. Finding a Mate (Sexual Selection)
Just like peacocks or birds of paradise, parrots use their feathers to attract mates. Brighter, more vibrant feathers are usually a sign of good health, strong genetics, and a robust diet. A parr